# Notebook 11 — Multi-Omics Integration (Survey)

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence | **This notebook: Tier 3 Survey**  
**Notebook:** 11 of 12  
**Time estimate:** 60 minutes

> No single omic layer tells the complete story.
> Integrating genomics, transcriptomics, and proteomics reveals regulatory
> mechanisms invisible to any one layer alone.
> This notebook surveys the landscape without implementing MOFA+ from scratch.

---
## Step 1 — Motivation

mRNA levels correlate only moderately with protein levels. Chromatin accessibility
correlates only partially with gene expression. Each omics layer captures different
aspects of cell state. Integration asks: what patterns are consistent across layers,
and what are layer-specific?

---
## Step 2 — Intuition

**Three integration strategies:**

1. **Early integration (feature concatenation):** Concatenate features from all
   omic layers into one matrix, then run standard analysis. Simple but ignores the
   different scales and noise structures of each layer.

2. **Late integration (separate analysis + combine results):** Analyse each layer
   independently, then compare results. E.g., find DE genes AND DA proteins and
   take the intersection. Misses cross-layer correlations.

3. **Model-based integration (MOFA+, Seurat WNN, DIABLO, LIGER):** Build a
   statistical model that jointly decomposes all layers. Identifies factors that
   drive variation across layers — revealing coordinated regulation.

**CITE-seq:** A single-cell technology that simultaneously measures mRNA (scRNA-seq)
and surface protein (antibody-derived tags, ADTs) in the same cell. The "same cell"
aspect eliminates cell-matching uncertainty — the gold standard for mRNA-protein
correlation at single-cell resolution.

---
## Step 3 — Biological Background

**MOFA+ (Multi-Omics Factor Analysis, Argelaguet 2018/2020):**  
A probabilistic factor analysis model. Factorizes each omic layer $X_m \approx Z W_m^T$
where $Z \in \mathbb{R}^{n \times K}$ are shared latent factors across cells and
$W_m$ are layer-specific loadings. Sparsity priors (ARD) drive factors to be active
in only a subset of layers.

**Seurat Weighted Nearest Neighbours (WNN):**  
In CITE-seq data, Seurat v4 computes a kNN graph in each modality separately, then
learns per-cell weights for each modality to construct a joint kNN graph. Cells in
regions where one modality is more informative get higher weight for that modality.

**LIGER (Linked Inference of Genomic Experimental Relationships):**  
Non-negative matrix factorization (NMF) integration. Decomposes paired datasets
into shared and dataset-specific factors. Used for scRNA-seq batch correction and
cross-modality integration.

**Key multi-omics datasets:**
- **CITE-seq PBMC (Stuart 2021):** gold standard for mRNA + protein validation
- **10x Multiome:** simultaneous scRNA-seq + scATAC-seq (chromatin accessibility)
- **REAP-seq:** RNA + epitope accessibility in single cells

---
## Step 4 — Mathematical Explanation

**MOFA+ model for view $m$:**
$$X_m = Z W_m^T + \epsilon_m, \quad \epsilon_m \sim \mathcal{N}(0, \tau_m^{-1}I)$$

where:
- $Z \in \mathbb{R}^{n \times K}$ — latent factor matrix (shared across views)
- $W_m \in \mathbb{R}^{p_m \times K}$ — view-specific loadings with ARD prior:
  $w_{mk} \sim \mathcal{N}(0, \alpha_{mk}^{-1}I)$
- $\alpha_{mk}$ — precision parameter; when large, factor $k$ is inactive in view $m$

Inference via variational Bayes. The trained model gives:
1. Factor scores $Z$ per sample — used for clustering/trajectory
2. Variance explained per factor per view — reveals which factors are view-specific
3. Feature weights — which genes/proteins drive each factor

**Simple correlation as baseline:**  
Before running MOFA+, always check cross-layer Pearson correlations:
$$r_{\text{mRNA-protein}} = \text{corr}(X_{\text{RNA}}, X_{\text{prot}})$$

Typically $r \approx 0.4$–$0.6$ genome-wide; much higher for specific pathways.

In [ ]:
# Step 6 — Python: Simulate matched multi-omics data + correlation analysis
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ---- Simulate matched mRNA + protein data for N samples ----
N_SAMPLES = 80
N_GENES = 300

# True latent biology (3 components)
Z = rng.normal(0, 1, (N_SAMPLES, 3))
# 3 groups of samples (e.g. cell types)
labels = rng.choice([0, 1, 2], N_SAMPLES)
for j in range(3):
    Z[labels == j, j] += 2.0

# mRNA: latent × loadings + noise
W_rna = rng.normal(0, 1, (N_GENES, 3))
W_rna[rng.random(N_GENES) < 0.7, :] = 0  # sparse loadings
RNA = Z @ W_rna.T + rng.normal(0, 1, (N_SAMPLES, N_GENES))

# Protein: shared latent + extra protein-specific noise (lower correlation)
# Only 50% of genes have matched protein data
N_PROTEINS = 150
W_prot = W_rna[:N_PROTEINS, :] + rng.normal(0, 0.5, (N_PROTEINS, 3))
PROT = Z @ W_prot.T + rng.normal(0, 1.5, (N_SAMPLES, N_PROTEINS))  # more noise

# ---- Cross-omic correlation ----
def cross_omics_correlation(RNA, PROT, n_genes_to_compare=None):
    """Pearson correlation per matched gene/protein pair."""
    n_match = PROT.shape[1]
    if n_genes_to_compare is not None:
        n_match = min(n_match, n_genes_to_compare)
    corrs = np.array([
        stats.pearsonr(RNA[:, g], PROT[:, g])[0]
        for g in range(n_match)
    ])
    return corrs

corrs = cross_omics_correlation(RNA, PROT)
print(f'mRNA-protein correlation:')
print(f'  Mean: {corrs.mean():.3f}')
print(f'  Median: {np.median(corrs):.3f}')
print(f'  Std: {corrs.std():.3f}')
print(f'  Fraction > 0.5: {(corrs > 0.5).mean()*100:.0f}%')
print(f'  Fraction < 0: {(corrs < 0).mean()*100:.0f}% (negative correlations)')

# ---- Simple early integration: PCA on concatenated data ----
def run_pca_2d(X):
    Xc = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return (U * s)[:, :2]

X_rna_sub   = RNA[:, :N_PROTEINS]   # matched genes
X_concat    = np.hstack([X_rna_sub, PROT])
Z_rna_only  = run_pca_2d(X_rna_sub)
Z_prot_only = run_pca_2d(PROT)
Z_concat    = run_pca_2d(X_concat)

print('\nPCA shapes:')
print(f'  RNA only:    {Z_rna_only.shape}')
print(f'  Protein only: {Z_prot_only.shape}')
print(f'  Concatenated: {Z_concat.shape}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
group_colors = {0: 'steelblue', 1: 'tomato', 2: 'forestgreen'}
c = [group_colors[l] for l in labels]

# Panel A: mRNA PCA
axes[0,0].scatter(Z_rna_only[:,0], Z_rna_only[:,1], c=c, s=20, alpha=0.7)
axes[0,0].set_title('A. RNA only (PCA2)')
axes[0,0].set_xlabel('PC1'); axes[0,0].set_ylabel('PC2')

# Panel B: Protein PCA
axes[0,1].scatter(Z_prot_only[:,0], Z_prot_only[:,1], c=c, s=20, alpha=0.7)
axes[0,1].set_title('B. Protein only (PCA2)')
axes[0,1].set_xlabel('PC1'); axes[0,1].set_ylabel('PC2')

# Panel C: Concatenated PCA
axes[0,2].scatter(Z_concat[:,0], Z_concat[:,1], c=c, s=20, alpha=0.7)
for g, col in group_colors.items():
    mask = labels == g
    axes[0,2].scatter([], [], c=col, label=f'Group {g}')
axes[0,2].legend(fontsize=8)
axes[0,2].set_title('C. Concatenated RNA+Protein (PCA2)')
axes[0,2].set_xlabel('PC1'); axes[0,2].set_ylabel('PC2')

# Panel D: mRNA-protein correlation histogram
axes[1,0].hist(corrs, bins=30, color='steelblue', alpha=0.8)
axes[1,0].axvline(corrs.mean(), color='red', ls='--', label=f'mean={corrs.mean():.2f}')
axes[1,0].set_xlabel('Pearson r (mRNA vs. protein)')
axes[1,0].set_ylabel('Number of genes')
axes[1,0].set_title('D. mRNA-protein correlation distribution')
axes[1,0].legend(fontsize=8)

# Panel E: Scatter for a high-correlation gene
top_g = np.argmax(corrs)
axes[1,1].scatter(RNA[:, top_g], PROT[:, top_g], c=c, s=15, alpha=0.7)
axes[1,1].set_xlabel('mRNA level')
axes[1,1].set_ylabel('Protein level')
axes[1,1].set_title(f'E. Best correlated gene (r={corrs[top_g]:.2f})')

# Panel F: Multi-omics tool overview (conceptual diagram)
ax = axes[1,2]
ax.axis('off')
tools = [
    ('MOFA+', 'Probabilistic NMF', 'Bulk or single-cell'),
    ('Seurat WNN', 'Weighted kNN graph', 'CITE-seq / Multiome'),
    ('LIGER', 'NMF', 'Batch correction'),
    ('DIABLO', 'Supervised', 'Paired multi-omics'),
]
y = 0.9
ax.text(0.5, 1.0, 'F. Multi-Omics Integration Tools', ha='center', fontsize=10, fontweight='bold')
for name, approach, use_case in tools:
    ax.text(0.05, y, f'{name}', fontsize=9, fontweight='bold')
    ax.text(0.35, y, f'{approach}', fontsize=8)
    ax.text(0.35, y - 0.05, f'Use: {use_case}', fontsize=7, color='grey')
    y -= 0.15

plt.suptitle('Module 10 NB11: Multi-Omics Integration Survey', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('multiomics_integration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. In the simulated data, compute the correlation for each of the 150 matched
   gene-protein pairs. Which pair has the highest and lowest correlation?
2. What is the difference between early integration and late integration? Give one
   advantage and one disadvantage of each.
3. Read the abstract of Argelaguet 2018 (MOFA). In one sentence: what biological
   problem was MOFA developed to address?
4. What is CITE-seq? Why does it give better mRNA-protein correlation data than
   bulk proteomics + bulk RNA-seq?

---
## Step 10 — Quiz

1. What are the three integration strategies and their main trade-offs?
2. What does a MOFA+ "factor" represent biologically?
3. Why is the typical mRNA-protein Pearson correlation only ~0.4–0.6?
4. What is CITE-seq and what does it measure simultaneously?
5. What is the key advantage of MOFA+ over simple PCA on concatenated data?

---
## Step 12 — Reflection

> *[In one paragraph: describe a biological question that cannot be answered by
> transcriptomics or proteomics alone, but could be answered by integrating both,
> and explain which integration method you would choose and why.]*

---
*Next: `12_mini_project_and_assessment.ipynb`*